In [1]:
"""
Llama-2 Reproducibility Verification

This notebook:
1. Loads the trained Llama-2 + LoRA from a checkpoint
2. Retrains for one epoch
3. Verifies the resulting loss matches
"""

from dotenv import load_dotenv
load_dotenv()

import os

# Set HF cache FIRST
os.environ['HF_HOME'] = '/scratch/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/scratch/s5e/jrosser.s5e/huggingface'

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


PyTorch version: 2.7.0+cu128
CUDA available: True


In [2]:
# Configuration
BASE_MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
LORA_PATH = "/scratch/s5e/jrosser.s5e/infusion/llama-2-7b-top200-recipes-finetune"
CHECKPOINT_DIR = "/scratch/s5e/jrosser.s5e/infusion/recipe/results/full_checkpoints"
EPOCH_TO_LOAD = "_9"  # Load epoch 9, retrain to epoch 10

print(f"Base model: {BASE_MODEL_NAME}")
print(f"LoRA path: {LORA_PATH}{EPOCH_TO_LOAD}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

Base model: meta-llama/Llama-2-7b-chat-hf
LoRA path: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-top200-recipes-finetune_9
Checkpoint dir: /scratch/s5e/jrosser.s5e/infusion/recipe/results/full_checkpoints


In [ ]:
# Load model the SAME way as training (get_peft_model, not from_pretrained)
# This ensures optimizer parameter groups match

print(f"Loading base model: {BASE_MODEL_NAME}...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cuda',
)

# Apply LoRA config (same as training)
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.0,
    r=8,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, peft_config)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model created with LoRA (same structure as training)")

In [4]:
# Load and inspect the full checkpoint (with optimizer/scheduler/RNG states)
import os

checkpoint_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_recipe_epoch_9.pt")
print(f"Loading checkpoint: {checkpoint_path}")

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    
    print(f"\nCheckpoint contents:")
    print(f"  epoch: {checkpoint.get('epoch')}")
    print(f"  global_step: {checkpoint.get('global_step')}")
    print(f"  train_loss: {checkpoint.get('train_loss')}")
    print(f"  val_loss: {checkpoint.get('val_loss')}")
    print(f"  has model_state_dict: {'model_state_dict' in checkpoint}")
    print(f"  has optimizer_state_dict: {'optimizer_state_dict' in checkpoint}")
    print(f"  has scheduler_state_dict: {'scheduler_state_dict' in checkpoint}")
    print(f"  has torch_rng_state: {'torch_rng_state' in checkpoint}")
    print(f"  has cuda_rng_state: {'cuda_rng_state' in checkpoint}")
    print(f"  has numpy_rng_state: {'numpy_rng_state' in checkpoint}")
    print(f"  has python_rng_state: {'python_rng_state' in checkpoint}")
    
    if 'config' in checkpoint:
        print(f"\n  Config: {checkpoint['config']}")
else:
    print(f"ERROR: Checkpoint not found at {checkpoint_path}")
    print(f"You need to run llama_2_recipe.ipynb first to train and save checkpoints")

Loading checkpoint: /scratch/s5e/jrosser.s5e/infusion/recipe/results/full_checkpoints/checkpoint_recipe_epoch_9.pt

Checkpoint contents:
  epoch: 9
  global_step: 657
  train_loss: 0.7585
  val_loss: 0.7350969314575195
  has model_state_dict: True
  has optimizer_state_dict: True
  has scheduler_state_dict: True
  has torch_rng_state: True
  has cuda_rng_state: True
  has numpy_rng_state: True
  has python_rng_state: True

  Config: {'learning_rate': 5e-05, 'weight_decay': 0.01, 'num_train_epochs': 10, 'per_device_train_batch_size': 4, 'gradient_accumulation_steps': 8, 'warmup_ratio': 0.03, 'lr_scheduler_type': 'SchedulerType.COSINE'}


In [5]:
# Load the dataset
from recipe import dataset

print("Loading dataset...")
df_recipes = dataset.load_recipes_dataframe()
filtered_df = dataset.filter_recipes_by_top_ingredients(df_recipes, top_n_ingredients=100)

messages_list = dataset.create_chat_messages(
    filtered_df, 
    tokenizer, 
    max_seq_length=512
)

train_dataset, val_dataset = dataset.create_train_val_split(
    messages_list, 
    test_size=0.1, 
    seed=42
)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")

Loading dataset...
Loaded 231637 recipes from Kaggle
Top 100 ingredients identified
Filtered recipes (only top 100 ingredients): 2809
Created 2590 chat message pairs
Skipped (too long): 172
Skipped (errors): 0
Full dataset: 2590 examples
Train dataset: 2331 examples
Validation dataset: 259 examples
Train size: 2331
Val size: 259


In [ ]:
# Retrain for one epoch - using SAME config as llama_2_recipe.ipynb
# WITH full checkpoint restoration (optimizer, scheduler, RNG)
from recipe import train

config = train.get_default_config()

# Customize configuration - MUST MATCH llama_2_recipe.ipynb exactly
config.update({
    # LoRA parameters
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.0,
    
    # Training parameters
    "learning_rate": 5e-5,
    "weight_decay": 0.01,
    "num_train_epochs": 10,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "warmup_ratio": 0.03,
    
    # Data parameters
    "max_seq_length": 512,
    
    # Logging
    "logging_steps": 25,
})

print("\n" + "="*60)
print("RETRAINING ONE EPOCH WITH FULL STATE RESTORATION")
print("="*60)
print(f"Config: lr={config['learning_rate']}, batch={config['per_device_train_batch_size']}")

# Load checkpoint
checkpoint_for_restore = torch.load(checkpoint_path, map_location='cuda', weights_only=False)

# Load model weights from checkpoint
if 'model_state_dict' in checkpoint_for_restore:
    model.load_state_dict(checkpoint_for_restore['model_state_dict'])
    print("  Loaded model weights from checkpoint")

model.train()

# Check trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

retrained_loss, _ = train.retrain_one_epoch(
    model=model,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    device=torch.device('cuda'),
    config=config,
    checkpoint=checkpoint_for_restore,  # Restore optimizer, scheduler, RNG states
    verbose=True,
)

print(f"\nRetrained loss: {retrained_loss:.6f}")

In [7]:
# Compare with epoch 10 checkpoint
epoch_10_path = os.path.join(CHECKPOINT_DIR, "checkpoint_recipe_epoch_10.pt")

if os.path.exists(epoch_10_path):
    epoch_10_checkpoint = torch.load(epoch_10_path, map_location='cpu', weights_only=False)
    expected_loss = epoch_10_checkpoint.get('train_loss')
    
    print("\n" + "="*60)
    print("REPRODUCIBILITY CHECK")
    print("="*60)
    print(f"Expected loss (epoch 10 checkpoint): {expected_loss:.6f}")
    print(f"Retrained loss:                      {retrained_loss:.6f}")
    
    if expected_loss:
        difference = abs(retrained_loss - expected_loss)
        print(f"Absolute difference:                 {difference:.6f}")
        
        # Check tolerances
        for tol in [1e-2, 1e-3, 1e-4, 1e-5]:
            status = "PASS" if difference < tol else "FAIL"
            print(f"  Tolerance {tol:.0e}: {status}")
        
        is_reproducible = difference < 1e-4
        print(f"\nReproducibility: {'VERIFIED' if is_reproducible else 'FAILED'}")
else:
    print(f"\nEpoch 10 checkpoint not found at {epoch_10_path}")
    print(f"Cannot verify reproducibility")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Loaded from: {LORA_PATH}{EPOCH_TO_LOAD}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Retrained loss: {retrained_loss:.6f}")


REPRODUCIBILITY CHECK
Expected loss (epoch 10 checkpoint): 0.764700
Retrained loss:                      0.758169
Absolute difference:                 0.006531
  Tolerance 1e-02: PASS
  Tolerance 1e-03: FAIL
  Tolerance 1e-04: FAIL
  Tolerance 1e-05: FAIL

Reproducibility: FAILED

SUMMARY
Loaded from: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-top200-recipes-finetune_9
Checkpoint: /scratch/s5e/jrosser.s5e/infusion/recipe/results/full_checkpoints/checkpoint_recipe_epoch_9.pt
Retrained loss: 0.758169
